# Collaborative Filtering from Scratch: Embeddings, Dot Products, and Bias

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/collaborative_filtering.ipynb)

Build a movie recommendation system from scratch using embedding dot products. Train on MovieLens 100k and explore the learned embeddings.

**Blog post:** [sesen.ai/blog/collaborative-filtering-embeddings-from-scratch](https://sesen.ai/blog/collaborative-filtering-embeddings-from-scratch)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, zipfile, urllib.request

## Load MovieLens 100k

In [ ]:
# Download MovieLens 100k if not present
data_dir = '/tmp/ml-100k'
if not os.path.exists(data_dir):
    url = 'https://files.grouplens.org/datasets/movielens/ml-100k.zip'
    zip_path = '/tmp/ml-100k.zip'
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/tmp/')
    print('Downloaded and extracted MovieLens 100k')

ratings = pd.read_csv(f'{data_dir}/u.data', delimiter='\t',
                       header=None, names=['user_id', 'movie_id', 'rating', 'timestamp'])
movies = pd.read_csv(f'{data_dir}/u.item', delimiter='|', encoding='latin-1',
                      header=None, usecols=[0, 1], names=['movie_id', 'title'])
ratings = ratings.merge(movies, on='movie_id')

n_users = ratings['user_id'].max()
n_movies = ratings['movie_id'].max()
print(f"{len(ratings)} ratings, {n_users} users, {n_movies} movies")
print(f"Sparsity: {1 - len(ratings) / (n_users * n_movies):.1%}")

## EmbeddingDotBias Model

In [ ]:
# Train/validation split
np.random.seed(42)
mask = np.random.rand(len(ratings)) < 0.9
train = ratings[mask].reset_index(drop=True)
valid = ratings[~mask].reset_index(drop=True)

class EmbeddingDotBias:
    """Collaborative filtering with embedding dot products and bias terms."""

    def __init__(self, n_users, n_items, n_factors=40, y_range=(0, 5.5)):
        self.y_range = y_range
        scale = 0.01
        # Embeddings: one vector per user and per item
        self.user_emb = np.random.randn(n_users + 1, n_factors) * scale
        self.item_emb = np.random.randn(n_items + 1, n_factors) * scale
        # Biases: one scalar per user and per item
        self.user_bias = np.zeros(n_users + 1)
        self.item_bias = np.zeros(n_items + 1)

    def predict(self, user_ids, item_ids):
        """Predict ratings: dot product of embeddings + biases, clamped to y_range."""
        dot = np.sum(self.user_emb[user_ids] * self.item_emb[item_ids], axis=1)
        pred = dot + self.user_bias[user_ids] + self.item_bias[item_ids]
        # Sigmoid to clamp to y_range (same as fast.ai EmbeddingDotBias)
        pred = self._sigmoid_range(pred)
        return pred

    def _sigmoid_range(self, x):
        lo, hi = self.y_range
        return lo + (hi - lo) * (1 / (1 + np.exp(-x)))

    def _sigmoid_range_grad(self, x):
        lo, hi = self.y_range
        s = 1 / (1 + np.exp(-x))
        return (hi - lo) * s * (1 - s)

    def train_step(self, user_ids, item_ids, ratings, lr=0.01, wd=0.1):
        """One training step with MSE loss, weight decay, and gradient descent."""
        # Forward pass
        dot = np.sum(self.user_emb[user_ids] * self.item_emb[item_ids], axis=1)
        raw = dot + self.user_bias[user_ids] + self.item_bias[item_ids]
        pred = self._sigmoid_range(raw)

        # MSE loss gradient
        error = pred - ratings  # (batch,)
        sig_grad = self._sigmoid_range_grad(raw)
        d_raw = error * sig_grad  # chain rule through sigmoid

        # Gradients for embeddings
        d_user_emb = d_raw[:, None] * self.item_emb[item_ids]
        d_item_emb = d_raw[:, None] * self.user_emb[user_ids]
        d_user_bias = d_raw
        d_item_bias = d_raw

        # Update with weight decay (L2 regularisation on embeddings)
        for uid, d_ue in zip(user_ids, d_user_emb):
            self.user_emb[uid] -= lr * (d_ue + wd * self.user_emb[uid])
        for iid, d_ie in zip(item_ids, d_item_emb):
            self.item_emb[iid] -= lr * (d_ie + wd * self.item_emb[iid])
        for uid, d_ub in zip(user_ids, d_user_bias):
            self.user_bias[uid] -= lr * d_ub
        for iid, d_ib in zip(item_ids, d_item_bias):
            self.item_bias[iid] -= lr * d_ib

        return np.mean(error**2)

In [ ]:
# Train
model = EmbeddingDotBias(n_users, n_movies, n_factors=40)
train_users = train['user_id'].values
train_items = train['movie_id'].values
train_ratings = train['rating'].values.astype(np.float32)
valid_users = valid['user_id'].values
valid_items = valid['movie_id'].values
valid_ratings = valid['rating'].values.astype(np.float32)

losses = []
for epoch in range(50):
    # Shuffle training data
    idx = np.random.permutation(len(train))
    epoch_loss = 0
    # Mini-batch training
    bs = 512
    for start in range(0, len(train), bs):
        batch = idx[start:start+bs]
        loss = model.train_step(train_users[batch], train_items[batch],
                                train_ratings[batch], lr=0.01, wd=0.1)
        epoch_loss += loss * len(batch)

    train_mse = epoch_loss / len(train)
    valid_pred = model.predict(valid_users, valid_items)
    valid_mse = np.mean((valid_pred - valid_ratings)**2)
    losses.append((train_mse, valid_mse))
    if epoch < 5 or (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:2d}: train MSE={train_mse:.4f}, valid MSE={valid_mse:.4f}")

## Training Curve

In [ ]:
train_losses, valid_losses = zip(*losses)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, 51), train_losses, 'o-', color='#3b82f6', linewidth=2, markersize=4, label='Train MSE')
ax.plot(range(1, 51), valid_losses, 'o-', color='#ef4444', linewidth=2, markersize=4, label='Valid MSE')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('MSE', fontsize=12)
ax.set_title('Collaborative Filtering: Training and Validation MSE', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Embeddings Are Just Lookup Tables

In [ ]:
# An embedding for 5 users in 3-dimensional space
user_embeddings = np.random.randn(5, 3)

# "Embed" user 2: just a lookup
user_2_vector = user_embeddings[2]
print(f"User 2 embedding: {user_2_vector}")

## The Dot Product: Measuring Similarity

In [ ]:
# User who likes action movies
user_vec = np.array([0.8, 0.1, -0.3])

# Two movies
action_movie = np.array([0.9, 0.2, -0.2])  # similar direction
romance_movie = np.array([-0.1, 0.7, 0.5])  # different direction

print(f"Action dot product:  {np.dot(user_vec, action_movie):.2f}")  # high = likes
print(f"Romance dot product: {np.dot(user_vec, romance_movie):.2f}")  # low = meh

## Movie Biases: Best and Worst Films

In [ ]:
# Get top movies by number of ratings
rating_counts = ratings.groupby('movie_id')['rating'].count()
top_movie_ids = rating_counts.sort_values(ascending=False).head(200).index.values

# Get bias and mean rating for top movies
movie_titles = movies.set_index('movie_id')['title']
movie_data = []
for mid in top_movie_ids:
    bias = model.item_bias[mid]
    mean_rating = ratings[ratings['movie_id'] == mid]['rating'].mean()
    movie_data.append((bias, movie_titles.get(mid, f'Movie {mid}'), mean_rating))

# Highest bias (best movies)
print("Highest bias (universally liked):")
for bias, title, mean in sorted(movie_data, key=lambda x: x[0], reverse=True)[:10]:
    print(f"  {bias:+.3f}  {mean:.2f}\u2605  {title}")

print("\nLowest bias (universally disliked):")
for bias, title, mean in sorted(movie_data, key=lambda x: x[0])[:10]:
    print(f"  {bias:+.3f}  {mean:.2f}\u2605  {title}")

In [ ]:
# Plot movie biases
sorted_data = sorted(movie_data, key=lambda x: x[0])
top10 = sorted_data[-10:]
bottom10 = sorted_data[:10]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

ax1.barh([d[1][:30] for d in bottom10], [d[0] for d in bottom10], color='#ef4444')
ax1.set_title('Lowest Bias (Disliked)', fontsize=12)
ax1.set_xlabel('Bias')

ax2.barh([d[1][:30] for d in top10], [d[0] for d in top10], color='#22c55e')
ax2.set_title('Highest Bias (Liked)', fontsize=12)
ax2.set_xlabel('Bias')

plt.suptitle('Learned Movie Biases (Top 200 Movies by Rating Count)', fontsize=14)
plt.tight_layout()
plt.show()

## PCA on Embeddings: Discovering Latent Dimensions

In [ ]:
from sklearn.decomposition import PCA

# Get embeddings for top-rated movies
top_embs = model.item_emb[top_movie_ids]
pca = PCA(n_components=2)
coords = pca.fit_transform(top_embs)

# Plot
fig, ax = plt.subplots(figsize=(12, 10))
ax.scatter(coords[:, 0], coords[:, 1], alpha=0.3)
for i, mid in enumerate(top_movie_ids[:50]):
    ax.annotate(movie_titles.get(mid, ''), (coords[i, 0], coords[i, 1]),
                fontsize=7, alpha=0.8)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
ax.set_title('Movie Embeddings (PCA, top 200 movies)', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Sparsity Visualisation

In [ ]:
# Visualise a small sample of the rating matrix
sample_users = np.arange(1, 51)
sample_movies = top_movie_ids[:50]

matrix = np.full((50, 50), np.nan)
for _, row in ratings.iterrows():
    if row['user_id'] in sample_users and row['movie_id'] in sample_movies:
        u_idx = row['user_id'] - 1
        m_idx = np.where(sample_movies == row['movie_id'])[0]
        if len(m_idx) > 0 and u_idx < 50:
            matrix[u_idx, m_idx[0]] = row['rating']

fig, ax = plt.subplots(figsize=(10, 8))
cmap = plt.cm.YlOrRd.copy()
cmap.set_bad(color='#f0f0f0')
im = ax.imshow(matrix, cmap=cmap, aspect='auto', vmin=1, vmax=5)
ax.set_xlabel('Movies (top 50 by rating count)', fontsize=12)
ax.set_ylabel('Users (first 50)', fontsize=12)
ax.set_title('Rating Matrix Sample (grey = missing)', fontsize=14)
plt.colorbar(im, label='Rating')
plt.tight_layout()
plt.show()

filled = np.sum(~np.isnan(matrix))
total = matrix.size
print(f"Filled: {filled}/{total} ({filled/total:.1%}), Missing: {1-filled/total:.1%}")

## Exercises

1. **No weight decay** — Train with `wd=0` and compare validation MSE. How much worse is overfitting?

2. **Embedding dimensions** — Try `n_factors=5`, `20`, `40`, `100`. Plot validation MSE vs factors. Where does increasing factors stop helping?

3. **Neural extension** — Concatenate user and movie embeddings and pass through a 2-layer MLP instead of using a dot product. Does validation MSE improve?

4. **Similar movies** — Pick a movie you like. Find the 5 nearest movies by cosine similarity of embeddings. Do the recommendations make sense?

5. **User clustering** — Apply K-Means to the user embeddings. Do the resulting clusters correspond to interpretable viewing preferences?

## References

- Koren, Y., Bell, R. & Volinsky, C. (2009). [Matrix Factorization Techniques for Recommender Systems.](https://datajobs.com/data-science-repo/Recommender-Systems-[Netflix].pdf) IEEE Computer, 42(8), 30-37.
- Simon Funk (2006). [Netflix Update: Try This at Home.](https://sifter.org/~simon/journal/20061211.html)
- fast.ai course: [Practical Deep Learning for Coders, Lesson 4](https://course.fast.ai/).